# Week 3: 统计推断与 Bootstrap

## 学习目标

1. 理解点估计和区间估计的概念
2. 掌握 Bootstrap 方法的基本原理
3. 学会计算置信区间
4. 理解假设检验的逻辑

## 1. 点估计

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("统计推断工具已加载")

### 1.1 样本统计量作为估计量

- **点估计**：用样本统计量估计总体参数
- **无偏性**：估计量的期望等于真值
- **一致性**：样本量增加时，估计量收敛到真值

In [ ]:
# 示例：估计总体均值
# 假设真实分布：N(μ=50, σ=10)

true_mu = 50
true_sigma = 10
population_size = 100000

# 生成总体（模拟）
population = np.random.normal(true_mu, true_sigma, population_size)

# 抽取样本
sample_size = 100
sample = np.random.choice(population, sample_size, replace=False)

# 点估计
sample_mean = np.mean(sample)
sample_std = np.std(sample, ddof=1)  # 无偏标准差

print("点估计结果")
print("=" * 40)
print(f"真实均值: {true_mu}")
print(f"样本均值: {sample_mean:.2f}")
print(f"估计误差: {abs(sample_mean - true_mu):.2f}")
print(f"\n真实标准差: {true_sigma}")
print(f"样本标准差: {sample_std:.2f}")

## 2. Bootstrap 方法

### 2.1 Bootstrap 原理

**核心思想**：用样本代替总体，通过有放回重采样估计统计量的分布。

**步骤**：
1. 从样本中有放回抽取 n 个观测
2. 计算统计量 θ*
3. 重复 B 次，得到 θ₁*, θ₂*, ..., θ_B*
4. 用这些值估计统计量的分布

In [ ]:
def bootstrap_sample(data, n_bootstraps=10000):
    """生成 Bootstrap 样本"""
    n = len(data)
    boot_samples = []
    
    for _ in range(n_bootstraps):
        # 有放回抽样
        boot = np.random.choice(data, size=n, replace=True)
        boot_samples.append(boot)
    
    return np.array(boot_samples)

# Bootstrap 估计均值
B = 10000
boot_samples = bootstrap_sample(sample, B)
boot_means = np.mean(boot_samples, axis=1)

print("Bootstrap 分析结果")
print("=" * 40)
print(f"Bootstrap 均值: {np.mean(boot_means):.2f}")
print(f"Bootstrap 标准误: {np.std(boot_means):.2f}")
print(f"理论标准误: {sample_std / np.sqrt(sample_size):.2f}")

In [ ]:
# 可视化 Bootstrap 分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 原始样本
axes[0].hist(sample, bins=20, density=True, alpha=0.7, label='样本')
axes[0].axvline(np.mean(sample), color='red', linestyle='--', label=f'样本均值 = {np.mean(sample):.2f}')
axes[0].axvline(true_mu, color='green', linestyle='-', label=f'真实均值 = {true_mu}')
axes[0].set_xlabel('值')
axes[0].set_ylabel('密度')
axes[0].set_title('原始样本分布')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bootstrap 均值分布
axes[1].hist(boot_means, bins=50, density=True, alpha=0.7, label='Bootstrap')
axes[1].axvline(np.mean(boot_means), color='red', linestyle='--', label='Bootstrap 均值')
axes[1].axvline(true_mu, color='green', linestyle='-', label='真实均值')
axes[1].set_xlabel('均值')
axes[1].set_ylabel('密度')
axes[1].set_title('Bootstrap 均值分布')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 置信区间

### 3.1 百分位数法

In [ ]:
def bootstrap_ci(data, stat_func=np.mean, alpha=0.05, n_bootstraps=10000):
    """用 Bootstrap 计算置信区间"""
    n = len(data)
    boot_stats = []
    
    for _ in range(n_bootstraps):
        boot = np.random.choice(data, size=n, replace=True)
        stat = stat_func(boot)
        boot_stats.append(stat)
    
    # 百分位数法
    lower = np.percentile(boot_stats, 100 * alpha/2)
    upper = np.percentile(boot_stats, 100 * (1 - alpha/2))
    
    return np.array(boot_stats), (lower, upper)

# 计算均值的 95% 置信区间
boot_means, ci_95 = bootstrap_ci(sample, np.mean, alpha=0.05)

print("95% 置信区间")
print("=" * 40)
print(f"Bootstrap 方法: [{ci_95[0]:.2f}, {ci_95[1]:.2f}]")

# 传统方法（假设正态分布）
se = sample_std / np.sqrt(sample_size)
t_critical = stats.t.ppf(0.975, df=sample_size-1)
traditional_ci = (sample_mean - t_critical * se, sample_mean + t_critical * se)
print(f"传统 t 方法: [{traditional_ci[0]:.2f}, {traditional_ci[1]:.2f}]")

print(f"\n真实均值 {true_mu} 是否在区间内？")
print(f"  Bootstrap: {ci_95[0] <= true_mu <= ci_95[1]}")
print(f"  传统方法: {traditional_ci[0] <= true_mu <= traditional_ci[1]}")

In [ ]:
# 可视化置信区间
fig, ax = plt.subplots(figsize=(12, 6))

# Bootstrap 分布
ax.hist(boot_means, bins=50, density=True, alpha=0.7, label='Bootstrap 分布')

# 置信区间
ax.axvline(ci_95[0], color='red', linestyle='--', linewidth=2, label='95% CI 下界')
ax.axvline(ci_95[1], color='red', linestyle='--', linewidth=2, label='95% CI 上界')
ax.axvline(true_mu, color='green', linestyle='-', linewidth=2, label='真实均值')
ax.axvline(sample_mean, color='orange', linestyle='-', linewidth=2, label='样本均值')

# 填充置信区间
ax.axvspan(ci_95[0], ci_95[1], alpha=0.2, color='red')

ax.set_xlabel('均值')
ax.set_ylabel('密度')
ax.set_title('Bootstrap 置信区间')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 中位数的置信区间

Bootstrap 的优势：不需要知道统计量的分布形式

In [ ]:
# 中位数的置信区间
boot_medians, median_ci = bootstrap_ci(sample, np.median, alpha=0.05)

true_median = np.median(population)
sample_median = np.median(sample)

print("中位数的 95% 置信区间")
print("=" * 40)
print(f"Bootstrap 方法: [{median_ci[0]:.2f}, {median_ci[1]:.2f}]")
print(f"样本中位数: {sample_median:.2f}")
print(f"真实中位数: {true_median:.2f}")
print(f"覆盖真实值: {median_ci[0] <= true_median <= median_ci[1]}")

## 4. 假设检验

### 4.1 Bootstrap 假设检验

In [ ]:
# 示例：检验两组需求是否有显著差异

# 假设数据：A 站点和 B 站点的日需求
demand_a = np.array([45, 52, 48, 51, 47, 53, 49, 50, 46, 48])
demand_b = np.array([38, 41, 39, 42, 37, 43, 40, 39, 41, 38])

print("两组站点需求对比")
print("=" * 40)
print(f"A 站点均值: {np.mean(demand_a):.2f}")
print(f"B 站点均值: {np.mean(demand_b):.2f}")
print(f"差异: {np.mean(demand_a) - np.mean(demand_b):.2f}")

In [ ]:
# Bootstrap 检验
def bootstrap_two_sample_test(sample1, sample2, n_bootstraps=10000):
    """两样本 Bootstrap 检验"""
    n1, n2 = len(sample1), len(sample2)
    observed_diff = np.mean(sample1) - np.mean(sample2)
    
    # 合并样本（零假设：两组来自同一分布）
    combined = np.concatenate([sample1, sample2])
    
    boot_diffs = []
    for _ in range(n_bootstraps):
        # 从合并样本中随机抽取
        boot1 = np.random.choice(combined, size=n1, replace=True)
        boot2 = np.random.choice(combined, size=n2, replace=True)
        boot_diff = np.mean(boot1) - np.mean(boot2)
        boot_diffs.append(boot_diff)
    
    boot_diffs = np.array(boot_diffs)
    
    # 双边检验 p 值
    p_value = np.mean(np.abs(boot_diffs) >= np.abs(observed_diff))
    
    return boot_diffs, observed_diff, p_value

boot_diffs, observed_diff, p_value = bootstrap_two_sample_test(demand_a, demand_b)

print("\nBootstrap 假设检验结果")
print("=" * 40)
print(f"观察到的差异: {observed_diff:.2f}")
print(f"p 值: {p_value:.4f}")
print(f"结论 (α=0.05): {'拒绝零假设' if p_value < 0.05 else '不能拒绝零假设'}")

In [ ]:
# 可视化
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(boot_diffs, bins=50, density=True, alpha=0.7, label='零假设下的差异分布')
ax.axvline(observed_diff, color='red', linestyle='--', linewidth=2, label=f'观察差异 = {observed_diff:.2f}')
ax.axvline(-observed_diff, color='red', linestyle='--', linewidth=2)

# 标记拒绝域
ax.axvspan(observed_diff, max(boot_diffs), alpha=0.3, color='red')
ax.axvspan(min(boot_diffs), -observed_diff, alpha=0.3, color='red')

ax.set_xlabel('均值差异')
ax.set_ylabel('密度')
ax.set_title('Bootstrap 假设检验')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 实际应用：需求分析

In [ ]:
# 读取实际数据
try:
    df = pd.read_csv('data/cleaned_bike_data.csv')
    
    # 分析 A001 站点的需求
    station_demand = df[df['station_id'] == 'A001']['demand'].values
    
    # Bootstrap 分析
    boot_means, mean_ci = bootstrap_ci(station_demand, np.mean)
    boot_stds, std_ci = bootstrap_ci(station_demand, np.std)
    
    print("A001 站点需求分析")
    print("=" * 40)
    print(f"样本量: {len(station_demand)}")
    print(f"样本均值: {np.mean(station_demand):.2f}")
    print(f"均值 95% CI: [{mean_ci[0]:.2f}, {mean_ci[1]:.2f}]")
    print(f"\n样本标准差: {np.std(station_demand):.2f}")
    print(f"标准差 95% CI: [{std_ci[0]:.2f}, {std_ci[1]:.2f}]")
    
except FileNotFoundError:
    print("数据文件未找到，使用模拟数据")

## 6. Research Thinking

### 问题 1：Bootstrap 的局限性

Bootstrap 什么时候可能失效？

**回答：**

1. **极端统计量**：如最大值、最小值
2. **小样本**：样本量 n < 20 时，Bootstrap 可能不稳定
3. **依赖结构**：时间序列数据需要特殊的 Bootstrap 方法
4. **重尾分布**：方差无限时，Bootstrap 不适用

### 问题 2：置信区间的解释

"95% 置信区间" 是什么意思？

**回答：**

- 不是 "真实值有 95% 概率在区间内"
- 而是 "如果重复抽样很多次，95% 的区间会包含真实值"
- 真实值是固定的，区间是随机的

### 问题 3：p 值的争议

为什么 p 值被滥用？更好的做法是什么？

**回答：**

**滥用问题：**
- p < 0.05 不等于 "有实际意义"
- 大样本下，微小差异也会显著
- p 值不能直接比较效应大小

**更好的做法：**
- 报告效应大小和置信区间
- 使用 Bootstrap 可视化不确定性
- 结合领域知识判断实际意义

## 7. 练习

### 练习 1
用 Bootstrap 方法计算样本方差的 95% 置信区间。

In [ ]:
# 你的代码


### 练习 2
比较 Bootstrap 置信区间和传统 t 检验置信区间在不同样本量下的表现。

In [ ]:
# 你的代码


### 练习 3
用 Bootstrap 方法分析：工作日和周末的需求差异是否显著？

In [ ]:
# 你的代码


## 8. 总结

### 本周学习要点

1. **点估计**：样本统计量估计总体参数
2. **Bootstrap**：有放回重采样估计统计量分布
3. **置信区间**：量化估计的不确定性
4. **假设检验**：Bootstrap 可以替代传统检验

### 关键洞察

- Bootstrap 不需要分布假设，适用范围广
- 置信区间比 p 值更有信息量
- 报告结果时应包含效应大小和置信区间